# 이벤트 전처리: 필수 이벤트만 남기기

윈도우로 나누기 **전에** 이벤트 단위로 정리한다.

```
build_labels.py → [이 노트북] → windows.py → features.py
event_labels.csv → events_A.csv / events_B.csv → windows_A_*.csv / windows_B_*.csv
```

| 단계 | 내용 |
|---|---|
| 1 | 서비스 이벤트 제외: AWS 서비스가 대신 호출한 이벤트 (EKS, Auto Scaling 등) |
| 2 | 반복 압축: 같은 주체가 같은 호출을 연달아 반복하면 1개로 합치고 반복 횟수를 남김 |
| 3 | 누출 제거: 라벨을 알려주는 필드(리전, userAgent, 이름, IP)를 일반화 |

두 가지 버전을 저장한다. 어느 쪽이 나은지는 실험으로 비교한다.

| 버전 | 반복 압축 | 이후 윈도우 단계 (`src/windows.py`) |
|---|---|---|
| A | 안 함 (서비스 제외만) | 분할한 뒤 내용이 같은 윈도우를 합친다 |
| B | 함 | 압축된 이벤트를 분할한다 |

누출 제거는 압축 전에 한 번만 해서 두 버전의 `actor`, `ip`, `region_changed` 값이 같게 한다.

규칙은 **라벨을 보지 않고** 이벤트 필드만으로 정한다. 실제 탐지 때도 같은 규칙을 쓸 수 있어야 하기 때문이다.
(예외: 반복 압축에서 공격·정상 이벤트가 하나로 합쳐지지 않도록 `attack_id`를 키에 넣는다.)

In [1]:
import ipaddress
import re
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IN = ROOT / "data" / "interim" / "event_labels.csv"
OUT_A = ROOT / "data" / "interim" / "events_A.csv"   # 압축 안 함
OUT_B = ROOT / "data" / "interim" / "events_B.csv"   # 반복 압축

CFG = {
    "drop_service_events": False,   # True면 AWS 서비스가 대신 호출한 이벤트를 뺀다
    "max_repeat_gap_sec": 300,   # 이 간격보다 멀리 떨어진 반복은 따로 둔다
}

In [2]:
events = pd.read_csv(IN)
events["eventTime"] = pd.to_datetime(events["eventTime"], utc=True)
events = events.sort_values(["session", "eventTime", "seq"], kind="stable").reset_index(drop=True)


def summary(df, name):
    n_att = int(df["is_attack"].sum())
    print(f"{name:12s} 이벤트 {len(df):6d}  공격 {n_att:5d}  정상 {len(df) - n_att:6d}  "
          f"기법 {df['technique'].nunique():2d}  세션 {df['session'].nunique()}")


summary(events, "원본")

원본           이벤트  28558  공격  8695  정상  19863  기법 29  세션 78


## 1. 서비스 이벤트 제외

AWS 서비스가 사용자를 대신해 호출한 이벤트는 사람이나 공격자의 행위가 아니다.

- `userType == "AWSService"` 또는 ARN에 `AWSServiceRole`
- 호출 출처(`sourceIPAddress`)와 `userAgent`가 **둘 다** `*.amazonaws.com` (예: `eks.amazonaws.com`)
  - 콘솔(`console.*`, `signin.*`)은 사람이 쓰는 것이므로 제외하지 않는다.

현재 설정은 `drop_service_events = False`로 **서비스 이벤트를 남긴다.** 정상 데이터를 늘리기 위한 실험이며, 서비스 이벤트 여부는 아래에서 표시만 한다.

In [3]:
def is_service_event(df):
    ip = df["sourceIPAddress"].fillna("")
    ua = df["userAgent"].fillna("")
    arn = df["userArn"].fillna("")
    via_service = ip.str.endswith(".amazonaws.com") & ua.str.endswith(".amazonaws.com") & ~ip.str.contains("console|signin")
    return df["userType"].eq("AWSService") | arn.str.contains("AWSServiceRole") | via_service


svc = is_service_event(events)
print("서비스 이벤트:", svc.sum())
print(events.loc[svc, "sourceIPAddress"].value_counts().head(8).to_string())
print("\n서비스 이벤트로 분류된 공격 이벤트:")
print(events.loc[svc & events["is_attack"], ["session", "eventName", "technique"]].to_string())

if CFG["drop_service_events"]:
    events = events[~svc].reset_index(drop=True)
summary(events, "서비스 제외" if CFG["drop_service_events"] else "서비스 포함")

서비스 이벤트: 12804
sourceIPAddress
eks.amazonaws.com               10711
autoscaling.amazonaws.com        1395
config.amazonaws.com              421
vpc-flow-logs.amazonaws.com        74
eks-nodegroup.amazonaws.com        69
lambda.amazonaws.com               63
ec2.amazonaws.com                  26
trustedadvisor.amazonaws.com       17

서비스 이벤트로 분류된 공격 이벤트:
                                     session    eventName  technique
26418  splunk:aws_invoke_model_access_denied  InvokeModel  T1087.004
서비스 포함       이벤트  28558  공격  8695  정상  19863  기법 29  세션 78


## 2. 반복 압축

같은 세션에서 **같은 주체**가 **같은 호출**(서비스, 이름, 에러코드)을 연달아 하면 첫 이벤트 1개만 남긴다.

- 다른 주체의 이벤트가 사이에 끼어 있어도 주체별 흐름 안에서 연속이면 합친다.
- 직전 같은 호출과 `max_repeat_gap_sec`보다 멀리 떨어지면 새 묶음으로 본다.
- 압축 전 정보는 `repeat_count`, `last_eventTime`, `duration_sec`로 남긴다. 윈도우 특징(`features.py`)에서 쓴다.

In [4]:
def compress_repeats(df, max_gap_sec):
    df = df.copy()
    actor = df["userArn"].fillna(df["userType"].fillna("unknown"))
    key = [df["session"], actor]
    call = (df["eventSource"].fillna("") + "|" + df["eventName"].fillna("") + "|"
            + df["errorCode"].fillna("") + "|" + df["attack_id"].fillna(""))

    prev_call = call.groupby(key).shift()
    gap = df["eventTime"].groupby(key).diff().dt.total_seconds()
    new_run = (call != prev_call) | (gap > max_gap_sec)
    run_id = new_run.groupby(key).cumsum()

    df["_run"] = df["session"] + "|" + actor + "|" + run_id.astype(str)
    agg = df.groupby("_run").agg(repeat_count=("_run", "size"), last_eventTime=("eventTime", "max"))
    first = df.drop_duplicates("_run").set_index("_run").join(agg)
    first["duration_sec"] = (first["last_eventTime"] - first["eventTime"]).dt.total_seconds()
    return first.reset_index(drop=True).sort_values(["session", "eventTime", "seq"], kind="stable").reset_index(drop=True)


before = events   # 버전 A의 바탕
events = compress_repeats(events, CFG["max_repeat_gap_sec"])   # 버전 B
summary(events, "반복 압축")

print("\n가장 많이 압축된 호출:")
print(events.nlargest(10, "repeat_count")[["session", "eventName", "is_attack", "repeat_count", "duration_sec"]].to_string())

반복 압축        이벤트   8303  공격  3515  정상   4788  기법 29  세션 78

가장 많이 압축된 호출:
                                               session               eventName  is_attack  repeat_count  duration_sec
3583                             splunk:aws_create_acl       DescribeInstances      False         10464       13650.0
3584                             splunk:aws_create_acl       DescribeInstances      False           227       13566.0
3899                             splunk:aws_create_acl  DescribeInstanceStatus      False           208        2758.0
4132                             splunk:aws_create_acl  DescribeInstanceStatus      False           204        2686.0
4610                             splunk:aws_create_acl  DescribeInstanceStatus      False           171        2231.0
3565  splunk:aws_cloudwatch_logs_high_volume_retrieval            GetLogEvents       True           168           8.0
6143       splunk:aws_iam_excessive_list_command_usage             ListObjects       True           

기법별 이벤트 수 변화. 이벤트가 많던 탐색 계열 기법이 크게 줄어 기법 간 불균형이 완화된다.

In [5]:
per_tech = pd.DataFrame({
    "원본": before[before["is_attack"]]["technique"].value_counts(),
    "압축 후": events[events["is_attack"]]["technique"].value_counts(),
}).fillna(0).astype(int).sort_values("원본", ascending=False)
per_tech

,원본,압축 후
technique,,
T1580,5391,1722
T1526,1071,939
T1530,759,57
T1496.001,594,367
T1098,245,179
T1078,126,99
T1651,85,8
T1552,60,2
T1552.GEN,53,6


## 3. 누출 제거

라벨을 정할 때는 썼지만 모델 입력에 들어가면 정답을 알려주는 필드들이다.

| 필드 | 문제 | 처리 |
|---|---|---|
| `awsRegion` | Stratus 리전은 `ap-central-2r`처럼 **가짜 리전**이라 리전만 보고 공격을 맞힐 수 있다 | 제거하고, 세션의 주 리전과 다른지만 `region_changed`로 남긴다 (T1535 신호) |
| `userAgent` | `stratus-red-team_<uuid>`. 도구 종류로 일반화해도 Stratus는 전부 Go SDK라 `sdk-go`만 보고 공격을 맞힐 수 있다 | **제거** (서비스 이벤트 판별에만 쓴다) |
| `userArn` | `rhino_escalate`, `stratus-red-team-...`, `cloudsploit` 같은 시나리오용 이름 | 세션 안에서만 의미 있는 가명(`IAMUser-1`)으로 바꾼다 |
| `sourceIPAddress` | 익명화 방식이 소스마다 다름 (Stratus는 `211.9.016.253`처럼 형식이 깨진 IP) | 종류(`ip_kind`)와 세션 내 가명(`ip-1`)으로 바꾼다 |
| `eventID`, `source_file`, `scenario` | 식별자 | 모델 입력에서 뺀다 (추적용으로 `event_uid`만 유지) |

In [6]:
def ip_kind(ip):
    if not isinstance(ip, str) or not ip:
        return "none"
    if ip.endswith(".amazonaws.com") or ip == "AWS Internal":
        return "aws-service"
    try:
        return "private" if ipaddress.ip_address(ip).is_private else "public"
    except ValueError:
        return "public" if re.fullmatch(r"[\d.]+|[0-9a-fA-F:]+", ip) else "other"


def pseudonym(df, col, prefix):
    """세션 안에서 처음 나온 순서대로 번호를 붙인다."""
    p = prefix if isinstance(prefix, pd.Series) else pd.Series(prefix, index=df.index)
    order = df.groupby("session")[col].transform(lambda s: pd.Series(pd.factorize(s)[0] + 1, index=s.index))
    return (p + "-" + order.astype(str)).where(df[col].notna())


def scrub(df):
    ev = df.copy()
    main_region = ev.groupby("session")["awsRegion"].transform(lambda s: s.mode().iat[0] if s.notna().any() else None)
    ev["region_changed"] = ev["awsRegion"].ne(main_region) & ev["awsRegion"].notna()
    ev["ip_kind"] = ev["sourceIPAddress"].map(ip_kind)
    ev["actor"] = pseudonym(ev, "userArn", ev["userType"].fillna("unknown"))
    ev["ip"] = pseudonym(ev, "sourceIPAddress", "ip")
    return ev


# 압축 전 이벤트에서 한 번만 계산하고, B는 A의 행 중 압축 후 남은 행을 가져온다
REPEAT_COLS = ["repeat_count", "last_eventTime", "duration_sec"]
ev_a = scrub(before).assign(repeat_count=1, last_eventTime=lambda d: d["eventTime"], duration_sec=0.0)
ev_b = events[["event_uid"] + REPEAT_COLS].merge(ev_a.drop(columns=REPEAT_COLS), on="event_uid", how="left")
assert len(ev_b) == len(events)

print(pd.crosstab(ev_a["ip_kind"], ev_a["source"]).to_string())

source       botsv3  splunk  stratus
ip_kind                             
aws-service     908   11950        0
other             0       2        0
private           0     677      108
public         5652    9058      203


남은 문자열 필드에 누출 문자열이 없는지 확인한다.

In [7]:
MODEL_COLS = [
    "event_uid", "session", "seq", "eventTime", "last_eventTime", "duration_sec", "repeat_count",
    "eventSource", "eventName", "readOnly", "errorCode",
    "userType", "actor", "ip_kind", "ip", "region_changed",
]
LABEL_COLS = ["source", "is_attack", "attack_id", "technique", "tactics", "label_level", "benign_type"]

out_a = ev_a[MODEL_COLS + LABEL_COLS]
out_b = ev_b[MODEL_COLS + LABEL_COLS]
leak = re.compile(r"stratus|red-team|rhino|cloudsploit|cloudmapper|atomic", re.I)
for name, out in [("A", out_a), ("B", out_b)]:
    model_text = out[[c for c in MODEL_COLS if c not in ("event_uid", "session")]].astype(str)
    hits = model_text.apply(lambda s: s.str.contains(leak)).sum()
    print(f"[{name}] 누출 문자열이 남은 필드:", hits[hits > 0].to_dict() or "없음")

[A] 누출 문자열이 남은 필드: 없음


[B] 누출 문자열이 남은 필드: 없음


In [8]:
for name, out, path in [("A (압축 안 함)", out_a, OUT_A), ("B (반복 압축)", out_b, OUT_B)]:
    out.to_csv(path, index=False)
    summary(out, name)
    print(f"  saved {path.relative_to(ROOT)}")

A (압축 안 함)   이벤트  28558  공격  8695  정상  19863  기법 29  세션 78
  saved data/interim/events_A.csv


B (반복 압축)    이벤트   8303  공격  3515  정상   4788  기법 29  세션 78
  saved data/interim/events_B.csv
